# Homework 02 — Generate User Stories, Test Cases, and API Specifications from a PRD

## Objective

Use the input document **`ABC_Health_Care_Outpatient_Scheduling_PRD.pdf`** and apply prompt engineering with **LangChain + OpenAI** to generate:

1. **User Stories with Acceptance Criteria**
2. **Test Cases**
3. **REST API Specifications / OpenAPI YAML**

## Technology
- Google Colab
- Python
- LangChain
- OpenAI
- PyPDF
- Colab Secrets

## Important design choice

This notebook uses **three specialized prompts** instead of one large prompt:

- Prompt 1 → User stories
- Prompt 2 → Test cases
- Prompt 3 → API specification

This makes the generated artifacts more focused, easier to validate, and easier to explain as a prompt-engineering homework.


## 1. Install Required Libraries

Run this cell once in Google Colab.


In [ ]:
!pip install -q langchain langchain-openai pypdf

## 2. Import Libraries and Load the OpenAI API Key

In Google Colab:

1. Click the **key icon** in the left sidebar.
2. Add a secret named `OPENAI_API_KEY`.
3. Enable notebook access.


In [ ]:
import os
import re
from pathlib import Path

from google.colab import userdata
from pypdf import PdfReader

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY was not found. Add it in Google Colab Secrets."
    )

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("OpenAI API key loaded successfully.")


## 3. Upload the PRD PDF

Upload the following file into the Colab session using the **Files** panel:

`ABC_Health_Care_Outpatient_Scheduling_PRD.pdf`

It should appear under `/content`.


In [ ]:
PDF_PATH = "/content/ABC_Health_Care_Outpatient_Scheduling_PRD.pdf"

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"PRD not found at {PDF_PATH}. "
        "Upload the PDF into the Colab Files panel first."
    )

print("PRD found:", PDF_PATH)


## 4. Extract Text from the PRD

The complete PRD text is extracted so the LLM can generate artifacts directly from the source requirements.

The prompts below explicitly instruct the model **not to invent unsupported requirements**.


In [ ]:
reader = PdfReader(PDF_PATH)

prd_pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    prd_pages.append(
        f"\n===== PAGE {page_number} =====\n{text}"
    )

prd_text = "\n".join(prd_pages)

print("Number of PDF pages:", len(reader.pages))
print("Extracted characters:", len(prd_text))

print("\nPRD preview:")
print(prd_text[:5000])
print("\n... preview truncated ...")


## 5. Initialize the OpenAI Model through LangChain

`temperature=0` is used because this homework requires structured, reproducible software artifacts rather than creative writing.

If the model below is unavailable in your OpenAI account, replace the model name with another supported OpenAI chat model.


In [ ]:
MODEL_NAME = "gpt-4.1-mini"

llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0
)

print("LangChain OpenAI model initialized:", MODEL_NAME)


# Part A — User Stories

## 6. Prompt Engineering for User Stories

This prompt asks the model to preserve the PRD's functional requirement IDs and generate implementation-ready user stories.

The model is asked to:
- use only the supplied PRD,
- keep traceability to requirement IDs,
- identify the actor,
- generate acceptance criteria,
- distinguish MVP vs optional/post-MVP items where the PRD makes that distinction.


In [ ]:
USER_STORIES_TEMPLATE = '''
You are a senior business analyst and product owner.

You are given the Product Requirements Document (PRD) for ABC Health Care
Company's Outpatient Patient Scheduling & Care System (OSCS).

Your task is to generate implementation-ready Agile user stories from the PRD.

STRICT RULES:
1. Use only information explicitly supported by the supplied PRD.
2. Do not invent workflows, policies, integrations, fields, compliance
   requirements, or business rules that are not present in the PRD.
3. Preserve the PRD's terminology and functional requirement IDs such as
   FR-001, FR-101, FR-201, etc.
4. Where the PRD states that a feature is optional, configurable, policy-gated,
   later-phase, or post-MVP, clearly mark that fact.
5. If the PRD does not provide enough detail for an acceptance criterion,
   write "Requires clarification" instead of inventing an answer.
6. Cover the important user personas in the PRD, including Patient,
   Front Desk/Scheduling Staff, Clinician, Nurse/Care Coordinator,
   Clinic Manager, System Admin/IT, and Billing Staff only where applicable.

For each story, produce:

### US-<number>: <short title>
- PRD Reference:
- Module:
- Persona:
- Priority / Scope: MVP, Optional MVP, Phase 2, or Not Explicitly Prioritized
- User Story:
  As a <persona>,
  I want <capability>,
  so that <business value>.
- Acceptance Criteria:
  - Given ...
    When ...
    Then ...
  - ...
- Dependencies / Notes:
- Clarifications Needed:

Organize the stories by PRD module:
A. Patient Portal and Access
B. Scheduling and Calendar Management
C. Intake, Check-in, and Clinic Flow
D. Care Management
E. Communication and Notifications
F. Reporting and Analytics
G. Administration and Configuration
H. Integrations

At the end, add a short "Traceability Summary" mapping each generated user story
to its source PRD requirement ID.

PRD:
----------------
{prd_text}
----------------
'''

user_stories_prompt = PromptTemplate(
    input_variables=["prd_text"],
    template=USER_STORIES_TEMPLATE
)

print("User stories prompt created.")


## 7. Generate User Stories


In [ ]:
user_stories_message = user_stories_prompt.format(
    prd_text=prd_text
)

user_stories_response = llm.invoke(user_stories_message)
user_stories = user_stories_response.content

print(user_stories)


## 8. Save User Stories


In [ ]:
OUTPUT_DIR = Path("/content/homework_02_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

user_stories_file = OUTPUT_DIR / "user_stories.md"
user_stories_file.write_text(user_stories, encoding="utf-8")

print("Saved:", user_stories_file)


# Part B — Test Cases

## 9. Prompt Engineering for Test Cases

This prompt generates test cases directly from the PRD.

It asks the model to cover:
- positive / happy path,
- negative validation,
- edge conditions,
- RBAC/security,
- audit logging,
- integration failures,
- performance requirements,
- appointment lifecycle/state transitions,
- MVP workflows.


In [ ]:
TEST_CASES_TEMPLATE = '''
You are a senior QA engineer designing a test suite for ABC Health Care
Company's Outpatient Patient Scheduling & Care System (OSCS).

Generate detailed software test cases from the supplied PRD.

STRICT RULES:
1. Use only requirements supported by the PRD.
2. Do not invent validation limits, response times, roles, business rules,
   status values, or external integrations.
3. Every test case must reference one or more PRD requirement IDs or a named
   non-functional requirement.
4. Preserve explicit requirements exactly, including:
   - appointment lifecycle states,
   - RBAC and least privilege,
   - PHI audit logging,
   - waitlist behavior,
   - scheduling eligibility,
   - overbooking justification,
   - availability search performance,
   - clinic status board update performance,
   - integration retry/alert behavior.
5. Where a precise expected result cannot be derived from the PRD, mark it
   "Requires clarification" instead of inventing a rule.

Generate a balanced suite containing:
- Functional positive tests
- Negative validation tests
- Boundary / edge cases
- Role-based access tests
- Audit-log tests
- Security tests supported by the PRD
- Integration failure/retry tests
- Performance tests
- Appointment state transition tests
- UAT-oriented end-to-end workflow tests

Use this format:

### TC-<number>: <test case title>
- PRD Reference:
- Module:
- Test Type:
- Priority:
- Preconditions:
- Test Data:
- Steps:
  1.
  2.
  3.
- Expected Result:
- Audit / Security Validation:
- Clarification Needed:

Also include a final coverage matrix with columns:

| PRD Requirement | Covered Test Cases | Coverage Type |

PRD:
----------------
{prd_text}
----------------
'''

test_cases_prompt = PromptTemplate(
    input_variables=["prd_text"],
    template=TEST_CASES_TEMPLATE
)

print("Test cases prompt created.")


## 10. Generate Test Cases


In [ ]:
test_cases_message = test_cases_prompt.format(
    prd_text=prd_text
)

test_cases_response = llm.invoke(test_cases_message)
test_cases = test_cases_response.content

print(test_cases)


## 11. Save Test Cases


In [ ]:
test_cases_file = OUTPUT_DIR / "test_cases.md"
test_cases_file.write_text(test_cases, encoding="utf-8")

print("Saved:", test_cases_file)


# Part C — API Specification

## 12. Prompt Engineering for REST API Design

The PRD defines product and functional requirements but does not prescribe a complete REST API.

Therefore, the prompt below instructs the model to create a **reasonable API design derived from the PRD**, while clearly avoiding unsupported business rules.

The output is requested as OpenAPI 3.x YAML.


In [ ]:
API_SPEC_TEMPLATE = '''
You are a senior API architect.

Using ONLY the supplied ABC Health Care Company Outpatient Patient Scheduling
& Care System (OSCS) PRD as the business-requirements source, design a REST API
specification suitable for implementing the documented workflows.

IMPORTANT:
The PRD describes functional requirements but does not prescribe exact REST
endpoint names or JSON schemas. Therefore:
- You may design reasonable resource-oriented REST endpoints and request/response
  schemas needed to implement the PRD.
- Clearly treat endpoint names and schema structures as implementation design,
  not as requirements quoted from the PRD.
- Do NOT invent new business rules.
- Preserve all explicit PRD business rules and state values.
- If a field or validation rule is not supported by the PRD, do not assign an
  arbitrary constraint.
- Include traceability using an extension named `x-prd-references` on operations
  where practical.

Generate a VALID OpenAPI 3.0.3 YAML document.

The API should cover the major MVP-relevant areas supported by the PRD:
1. Patient registration/profile
2. Appointment availability search
3. Appointment request/booking
4. Reschedule/cancel
5. Appointment state lifecycle
6. Provider calendars / scheduling
7. Waitlist
8. Intake forms / consents
9. Check-in
10. Care plans and tasks
11. Notifications
12. Clinic status board
13. Reporting
14. Administration/configuration where appropriate
15. EHR/EMR integration touchpoints where representable as application APIs
16. Audit-related behavior where appropriate

Include:
- `openapi: 3.0.3`
- info
- servers
- tags
- paths
- HTTP methods
- path/query parameters
- request bodies
- response bodies
- appropriate common HTTP status codes
- security scheme(s)
- reusable schemas under components
- the exact appointment states from the PRD:
  Requested, Scheduled, Confirmed, Checked-in, In-room, Completed,
  Cancelled, No-show

For error responses, define a reusable generic error schema.

For PHI-sensitive and role-protected operations, document that authorization
is required. Do not claim specific OAuth scopes unless the PRD explicitly
defines them.

Return ONLY YAML.
Do not wrap the YAML in Markdown code fences.
Do not add commentary before or after it.

PRD:
----------------
{prd_text}
----------------
'''

api_spec_prompt = PromptTemplate(
    input_variables=["prd_text"],
    template=API_SPEC_TEMPLATE
)

print("API specification prompt created.")


## 13. Generate OpenAPI YAML


In [ ]:
api_spec_message = api_spec_prompt.format(
    prd_text=prd_text
)

api_spec_response = llm.invoke(api_spec_message)
api_spec_yaml = api_spec_response.content.strip()

# Remove Markdown fences if the model adds them despite the instruction.
api_spec_yaml = re.sub(
    r"^```(?:yaml|yml)?\s*",
    "",
    api_spec_yaml,
    flags=re.IGNORECASE
)
api_spec_yaml = re.sub(r"\s*```$", "", api_spec_yaml)

print(api_spec_yaml[:8000])
print("\n... output may be truncated in notebook display ...")


## 14. Save the OpenAPI Specification


In [ ]:
api_spec_file = OUTPUT_DIR / "openapi.yaml"
api_spec_file.write_text(api_spec_yaml, encoding="utf-8")

print("Saved:", api_spec_file)


# Part D — Basic Output Validation

## 15. Verify that the Expected Artifacts Were Generated

This does not prove that every LLM output is correct, but it catches basic failures such as empty output or a missing OpenAPI header.


In [ ]:
print("User stories generated:", bool(user_stories.strip()))
print("Test cases generated:", bool(test_cases.strip()))
print("API spec generated:", bool(api_spec_yaml.strip()))

assert user_stories.strip(), "User stories output is empty."
assert test_cases.strip(), "Test cases output is empty."
assert api_spec_yaml.strip(), "OpenAPI output is empty."

if "openapi: 3.0.3" in api_spec_yaml:
    print("OpenAPI version header found.")
else:
    print(
        "WARNING: Expected 'openapi: 3.0.3' header was not found. "
        "Review the generated API output."
    )


## 16. Optional: Preview the Prompt Text

These cells are useful when presenting the homework because they show that the solution is based on **prompt engineering**, not just API calls.


In [ ]:
print("USER STORIES PROMPT PREVIEW")
print("=" * 80)
print(user_stories_message[:6000])
print("\n... prompt truncated ...")


# 17. Generated Files

After all cells run successfully, the notebook creates:

```text
/content/homework_02_outputs/
├── user_stories.md
├── test_cases.md
└── openapi.yaml
```

You can download these files from the Colab Files panel and commit them to GitHub.


In [ ]:
print("Generated files:")

for file_path in OUTPUT_DIR.iterdir():
    print("-", file_path)


# 18. Conclusion

This homework uses Generative AI to convert a Product Requirements Document into software-development artifacts.

### Prompt Engineering Approach

Instead of using one broad prompt, three task-specific prompts were created:

1. **Business Analysis Prompt**
   - Generates user stories and acceptance criteria.
   - Preserves PRD requirement traceability.

2. **QA Prompt**
   - Generates functional, negative, edge, security, RBAC, audit, integration, and performance tests.
   - Includes a requirements coverage matrix.

3. **API Architecture Prompt**
   - Converts the business requirements into a proposed REST interface.
   - Produces an OpenAPI 3.x YAML specification.
   - Distinguishes implementation design choices from explicit PRD business rules.

### Important Observation

Generative AI can accelerate requirements analysis and documentation, but generated artifacts still require human review.

For this assignment, the prompts are intentionally written to prevent the model from silently inventing requirements. If the PRD does not support a specific business rule, the model is instructed to identify the gap instead of fabricating one.


# 19. Suggested GitHub Structure

```text
Homework_02_Healthcare_PRD/
│
├── Homework_02_Healthcare_PRD.ipynb
├── ABC_Health_Care_Outpatient_Scheduling_PRD.pdf
│
├── outputs/
│   ├── user_stories.md
│   ├── test_cases.md
│   └── openapi.yaml
│
└── README.md
```

### Security

Do **not** place your OpenAI API key in the notebook or GitHub repository.

Keep the key in **Google Colab Secrets** using:

`OPENAI_API_KEY`
